# 🏭 Project 4.1 — Conveyor Belt Inspector
**DSA for Mechatronics · Week 04 — Linked Lists**

> **Run all cells top to bottom. Submit the .ipynb on Blackboard.**
> The final cell prints your personalised report — be ready to explain every step.


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║            ENTER YOUR STUDENT ID BELOW              ║
# ╚══════════════════════════════════════════════════════╝

STUDENT_ID = "12345678"   # ← replace with your real student ID

import random, hashlib
_seed = int(hashlib.md5(STUDENT_ID.encode()).hexdigest(), 16) % (2**31)
random.seed(_seed)
print(f"Student ID  : {STUDENT_ID}")
print(f"Dataset seed: {_seed}")
print("✅ Your personal dataset is ready.")


## The Scenario
A conveyor belt carries numbered parts past a quality sensor.
Parts are added at the **tail** (arrival) and occasionally pulled from any position when flagged.
The belt is modelled as a **singly linked list** — insertions at the tail are frequent, random access is never needed.

## Step 1 — Define the Node and build your belt

In [ ]:
# ── Node class ─────────────────────────────────────────────────────
class Node:
    """One part on the conveyor belt."""
    def __init__(self, part_id, quality):
        self.part_id  = part_id          # unique part number
        self.quality  = quality          # 0–100  (below THRESHOLD = faulty)
        self.next     = None

    def __repr__(self):
        return f"Part({self.part_id}, q={self.quality})"

# ── Personalised belt parameters ───────────────────────────────────
BELT_LEN       = random.randint(18, 30)
THRESHOLD      = random.randint(30, 55)       # quality below this = faulty
FAULT_PROB     = round(random.uniform(0.15, 0.35), 2)
PART_ID_START  = random.randint(1000, 2000)

# ── Build the belt as a singly linked list ──────────────────────────
head = None
tail = None
part_ids = random.sample(range(PART_ID_START, PART_ID_START + BELT_LEN * 3), BELT_LEN)

for pid in part_ids:
    quality  = random.randint(0, 40) if random.random() < FAULT_PROB else random.randint(55, 100)
    new_node = Node(pid, quality)
    if head is None:
        head = tail = new_node
    else:
        tail.next = new_node
        tail = new_node

def belt_to_list(h):
    out = []
    cur = h
    while cur:
        out.append(cur)
        cur = cur.next
    return out

parts = belt_to_list(head)
n_faulty = sum(1 for p in parts if p.quality < THRESHOLD)

print(f"Belt parameters:")
print(f"  Parts on belt  : {BELT_LEN}")
print(f"  Quality threshold: {THRESHOLD}  (below = faulty)")
print(f"  Fault probability: {FAULT_PROB*100:.0f}%")
print(f"  Faulty parts now : {n_faulty}")
print()
print(f"  {'#':>5}  {'Part ID':>8}  {'Quality':>8}  Status")
print(f"  {'─'*5}  {'─'*8}  {'─'*8}  {'─'*10}")
for i, p in enumerate(parts):
    status = "⚠ FAULTY" if p.quality < THRESHOLD else "OK"
    print(f"  {i:5d}  {p.part_id:8d}  {p.quality:8d}  {status}")


## Step 2 — Traversal: count, search, statistics

In [ ]:
def traverse_belt(head):
    """
    Walk the entire list once, collecting:
      - total count
      - number of faulty parts (quality < THRESHOLD)
      - minimum and maximum quality
      - list of faulty part IDs
    Returns a dict of results.
    """
    count = 0
    faulty_count = 0
    faulty_ids   = []
    min_q = 101
    max_q = -1
    total_q = 0

    current = head
    while current:
        count    += 1
        total_q  += current.quality
        if current.quality < min_q: min_q = current.quality
        if current.quality > max_q: max_q = current.quality
        if current.quality < THRESHOLD:
            faulty_count += 1
            faulty_ids.append(current.part_id)
        current = current.next

    return {
        "count":        count,
        "faulty_count": faulty_count,
        "faulty_ids":   faulty_ids,
        "min_quality":  min_q,
        "max_quality":  max_q,
        "avg_quality":  round(total_q / count, 1) if count else 0,
    }

stats = traverse_belt(head)

print("Belt traversal results:")
print(f"  Total parts    : {stats['count']}")
print(f"  Faulty parts   : {stats['faulty_count']}  ({stats['faulty_count']/stats['count']*100:.1f}%)")
print(f"  Quality range  : {stats['min_quality']} – {stats['max_quality']}")
print(f"  Average quality: {stats['avg_quality']}")
print(f"  Faulty IDs     : {stats['faulty_ids']}")


## Step 3 — Deletion: remove all faulty parts (dummy head pattern)

In [ ]:
def remove_faulty(head, threshold):
    """
    Remove all nodes where quality < threshold.
    Uses the dummy head pattern to handle head-removal cleanly.
    Returns the new head.
    """
    dummy = Node(-1, 999)       # sentinel node before real list
    dummy.next = head

    prev    = dummy
    current = dummy.next

    removed = []
    while current:
        if current.quality < threshold:
            removed.append(current.part_id)
            prev.next = current.next    # skip the faulty node
        else:
            prev = current              # only advance prev when keeping
        current = current.next

    return dummy.next, removed

head, removed_ids = remove_faulty(head, THRESHOLD)
remaining = belt_to_list(head)

print(f"After removing faulty parts (threshold = {THRESHOLD}):")
print(f"  Parts removed  : {len(removed_ids)}")
print(f"  Removed IDs    : {removed_ids}")
print(f"  Parts remaining: {len(remaining)}")
print()
print(f"  {'#':>5}  {'Part ID':>8}  {'Quality':>8}")
print(f"  {'─'*5}  {'─'*8}  {'─'*8}")
for i, p in enumerate(remaining):
    print(f"  {i:5d}  {p.part_id:8d}  {p.quality:8d}")


## Step 4 — Fast/Slow pointers: find the median-quality part

In [ ]:
def find_middle(head):
    """
    Find the middle node using fast/slow pointers.
    Fast moves 2 steps, slow moves 1 — when fast reaches end, slow is at middle.
    O(n) time, O(1) space.
    """
    if not head:
        return None
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

def find_nth_from_end(head, n):
    """
    Find nth node from the end using two pointers.
    Lead pointer starts n steps ahead; when lead hits end, trail is at target.
    """
    lead  = head
    trail = head
    for _ in range(n):
        if lead is None:
            return None       # list shorter than n
        lead = lead.next
    while lead:
        lead  = lead.next
        trail = trail.next
    return trail

middle = find_middle(head)
n_check = random.randint(1, max(1, len(remaining) - 1))
nth     = find_nth_from_end(head, n_check)

print("Fast/Slow pointer results:")
print(f"  Belt length    : {len(remaining)} parts")
print(f"  Middle part    : Part {middle.part_id}  (quality {middle.quality})")
print(f"  {n_check}-from-end part : Part {nth.part_id}  (quality {nth.quality})" if nth else f"  {n_check}-from-end: out of range")


## Step 5 — Reversal: reverse the belt order in-place

In [ ]:
def reverse_list(head):
    """
    Reverse the linked list in-place.
    Three-pointer iterative approach: prev, current, next_node.
    O(n) time, O(1) space.
    """
    prev    = None
    current = head
    while current:
        next_node   = current.next   # save next
        current.next = prev          # reverse link
        prev         = current       # advance prev
        current      = next_node     # advance current
    return prev   # prev is now the new head

head_before = [p.part_id for p in belt_to_list(head)]
head = reverse_list(head)
head_after  = [p.part_id for p in belt_to_list(head)]

print("Belt reversed (last part is now first at the sensor):")
print(f"  Before: {head_before}")
print(f"  After : {head_after}")


## 📋 Final Report

In [ ]:
W = 56
clean_len    = len(remaining)
fault_rate   = stats["faulty_count"] / stats["count"] * 100
status       = "🔴 HIGH FAULT RATE" if fault_rate > 25 else ("🟡 MODERATE" if fault_rate > 10 else "🟢 NORMAL")

print("╔" + "═"*W + "╗")
print("║" + " CONVEYOR BELT INSPECTION REPORT".center(W) + "║")
print("╠" + "═"*W + "╣")
print(f"║  {'Student ID':<26}: {STUDENT_ID:<26}║")
print(f"║  {'Dataset seed':<26}: {_seed:<26}║")
print("╠" + "═"*W + "╣")
print("║  BELT PARAMETERS" + " "*(W-17) + "║")
print(f"║  {'Initial belt length':<26}: {BELT_LEN:<26}║")
print(f"║  {'Quality threshold':<26}: {THRESHOLD:<26}║")
print(f"║  {'Fault probability':<26}: {FAULT_PROB*100:.0f}%{'':<24}║")
print("╠" + "═"*W + "╣")
print("║  INSPECTION RESULTS" + " "*(W-20) + "║")
print(f"║  {'Parts inspected':<26}: {stats['count']:<26}║")
print(f"║  {'Faulty parts found':<26}: {stats['faulty_count']}  ({fault_rate:.1f}%){'':<16}║")
print(f"║  {'Quality range':<26}: {stats['min_quality']} – {stats['max_quality']}{'':<19}║")
print(f"║  {'Average quality':<26}: {stats['avg_quality']:<26}║")
print(f"║  {'Parts after cleaning':<26}: {clean_len:<26}║")
print("╠" + "═"*W + "╣")
print("║  POINTER OPERATIONS" + " "*(W-20) + "║")
print(f"║  {'Middle part ID':<26}: {middle.part_id}  (quality {middle.quality}){'':<8}║")
print(f"║  {str(n_check)+'-from-end part ID':<26}: {nth.part_id if nth else 'N/A':<26}║")
print("╠" + "═"*W + "╣")
print(f"║  STATUS : {status:<{W-11}}║")
print("╚" + "═"*W + "╝")


---
## 📝 Reflection — answer before submitting

Double-click this cell to edit. Write 2–4 sentences for each question.

---

**Q1 — What does your output show?**
Describe the key results from your final report. What does it tell you about the system?

*Your answer here:*

---

**Q2 — Which linked list concept did you find most important, and why?**
Pick one technique from the notebook (fast/slow pointers, dummy head, reversal, cycle detection…) and explain in your own words what problem it solves.

*Your answer here:*

---

**Q3 — What would change if your student ID changed?**
Which specific numbers or results in the final report would be different, and why?

*Your answer here:*
